# Configurando o Ambiente

**Configuração Modelo do Autor** 
- formula = "group ~ C(Island, Treatment(reference='Gran Canaria')) + C(Breed, Treatment(reference='Crossbreed'))"
- modelo = smf.logit(formula, data=study_filtrado).fit(disp=0)

**Sobre a Fórmula Utilizada:**

Ajuste de Variáveis (Confounding): Ao colocar Island e Breed juntas, o autor isola o efeito da raça. Isso impede que o modelo atribua um risco alto a uma raça apenas porque ela é muito comum em uma ilha que possui fatores ambientais de risco (como poluição ou dieta regional).

Estabelecimento de Baseline: O comando Treatment(reference=...) define que o "Cachorro Zero" do estudo é o Crossbreed (SRD) vivendo em Gran Canaria. Todos os outros resultados são calculados como uma variação (maior ou menor) em relação a esse padrão.

**Desempenho do Modelo do Autor (Baseline)**
- AUC-ROC: 0.789 (Quão bem ele separa doentes de saudáveis)
- Brier Score: 0.025 (Quão bem calibrada é a porcentagem. Mais baixo é melhor)

### Analisando Base Utilizada

In [2]:
import pandas as pd
study_filtrado = pd.read_csv('../data/study_filtered.csv')
study_filtrado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76618 entries, 0 to 76617
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Breed   76618 non-null  object
 1   Island  76618 non-null  object
 2   group   76618 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.8+ MB


In [2]:
study_filtrado['group'].value_counts()

group
0    74601
1     2017
Name: count, dtype: int64

In [3]:
study_filtrado['Breed'].value_counts()

Breed
Crossbreed                        27324
Canarian Warren Hound             14999
Yorkshire Terrier                  6902
Chihuahua                          4006
French Bulldog                     3177
Labrador Retriever                 1865
English Pointer                    1772
German Shepherd Dog                1610
Canarian Mastiff                   1267
Pit Bull Terrier                   1140
Cocker Spaniel                      977
Poodle                              936
Miniature Pinscher                  900
Staffordshire Bull Terrier          791
Golden Retriever                    753
Bichon frise                        702
Boxer                               685
American Staffordshire Terrier      633
Majorero                            621
Bull Terrier                        564
Shih-Tzu                            450
Jack Russell Terrier                428
Andalusian Ratter                   428
Dalmatian                           413
Beagle                            

### Preparando Dados

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

dados_ml = pd.get_dummies(study_filtrado, columns=['Breed', 'Island'], drop_first=True)

X = dados_ml.drop(columns=['group'])
y = dados_ml['group']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [12]:
def avaliar_modelo(modelo, X_test, y_test, nome):
    probs = modelo.predict_proba(X_test)[:, 1]  
    auc = roc_auc_score(y_test, probs)
    brier = brier_score_loss(y_test, probs)
    
    print(f"{nome} -> AUC-ROC: {auc:.4f} | Brier Score: {brier:.4f}")
    #return auc, brier

# Modelos

## Regressão Classica (sem penalidade)

### Modelo Base

In [6]:
import statsmodels.formula.api as smf

formula = "group ~ C(Island, Treatment(reference='Gran Canaria')) + C(Breed, Treatment(reference='Crossbreed'))"
smf_model = smf.logit(formula, data=study_filtrado).fit(disp=0)

probabilidades_autor = smf_model.predict(study_filtrado)

target = study_filtrado['group']

auc_autor = roc_auc_score(target, probabilidades_autor)
brier_autor = brier_score_loss(target, probabilidades_autor)

print("--- Desempenho do Modelo do Autor (Baseline) ---")
print(f"AUC-ROC: {auc_autor:.4f}")
print(f"Brier Score: {brier_autor:.4f}")

--- Desempenho do Modelo do Autor (Baseline) ---
AUC-ROC: 0.7895
Brier Score: 0.0246


### Modelo Base Divisão 80/20

In [7]:
df_treino, df_teste = train_test_split(
    study_filtrado, 
    test_size=0.2, 
    random_state=42, 
    stratify=study_filtrado['group']
)

formula = "group ~ C(Island, Treatment(reference='Gran Canaria')) + C(Breed, Treatment(reference='Crossbreed'))"

smf_model_split = smf.logit(formula, data=df_treino).fit(disp=0)
probs = smf_model_split.predict(df_teste)

target = df_teste['group']

auc = roc_auc_score(target, probs)
brier = brier_score_loss(target, probs)

print("--- Desempenho do Modelo do Autor (Split 80/20) ---")
print(f"AUC-ROC: {auc:.4f}")
print(f"Brier Score: {brier:.4f}")

--- Desempenho do Modelo do Autor (Split 80/20) ---
AUC-ROC: 0.7624
Brier Score: 0.0249


### Scikit Learn

#### Parâmetros

- penalty -> penalidade aplicada às variáveis
    - l2 -> Encolhe os pesos de todas as variáveis, mas não zera nenhuma. Ótimo para manter todas as raças no modelo de forma equilibrada.
    - l1 -> Zera os pesos das variáveis que não importam. Perfeito para "limpar" o modelo e manter só as raças cruciais.
    - elasticnet -> Mistura L1 e L2
    - None -> sem penalidade

- C -> É a força inversa da regularização
    - Valores altos: Menos restrição. O modelo confia muito nos dados de treino.
    - Valores baixos: Mais restrição. O modelo fica mais cético, impedindo que raças com poucos casos dominem a probabilidade.

- l1_ratio -> Só é usado se você escolher penalty='elasticnet'. Ele dita a proporção da mistura.
    - Um valor de 0.2 significa 20% Lasso (zerar variáveis) e 80% Ridge (encolher variáveis). Um valor de 0.5 é meio a meio.

- solver -> Define a engrenagem de cálculo.
    - 'lbfgs': Excelente e rápido para a maioria dos casos, mas só suporta penalidade L2 ou None.asd
    - 'liblinear': O melhor para datasets menores e é obrigatório se você quiser usar a penalidade L1 (Lasso).
    - 'saga': O único que suporta Elastic-Net. Também é muito rápido para datasets gigantescos (centenas de milhares de linhas).
    - newton-cholesky: Ele calcula a matriz de curvatura exata e a resolve usando uma técnica de álgebra linear chamada "Decomposição de Cholesky".

- class_weight -> Dá pesos diferentes para acertos e erros dependendo da classe.
    - None: Trata todos os cães com o mesmo peso.
    - 'balanced': O modelo matematicamente aumenta o "peso" do erro quando ele erra um cão com câncer. Ele força o algoritmo a prestar muito mais atenção na minoria doente para equilibrar o jogo.

- max_iter -> O número máximo de tentativas (passos) que o solver tem para achar a matemática perfeita.
    - Se você rodar o modelo e receber um aviso vermelho dizendo "ConvergenceWarning: lbfgs failed to converge", significa que o modelo precisava de mais tempo para pensar. Aumente para 500 ou 1000.


In [8]:
classic_lbfgs = LogisticRegression(
    penalty=None,
    solver='lbfgs', 
    class_weight='balanced', 
    random_state=42
)

classic_lbfgs.fit(X_train, y_train)
auc, brier = avaliar_modelo(classic_lbfgs, X_test, y_test, "Logistic Regression SP (LBFGS)")


Logistic Regression SP (LBFGS) -> AUC-ROC: 0.7624 | Brier Score: 0.1974


In [9]:
classic_lbfgs2 = LogisticRegression(
    penalty=None, 
    solver='newton-cg', 
    class_weight='balanced', 
    random_state=42
)

classic_lbfgs2.fit(X_train, y_train)
auc, brier = avaliar_modelo(classic_lbfgs2, X_test, y_test, "Logistic Regression SP (Newton-CG)")

Logistic Regression SP (Newton-CG) -> AUC-ROC: 0.7630 | Brier Score: 0.1974


## Regressão Logistica ML (com penalidade)

### Lasso

In [10]:
def try_logistic_regression(penalty='l1', C=None, class_weight='balanced', solver='liblinear', max_iter=100):
    model = LogisticRegression(
        penalty=penalty, 
        C=C, 
        solver=solver, 
        class_weight=class_weight, 
        max_iter=max_iter, 
        random_state=42
    )
    model.fit(X_train, y_train)
    print(f"Logistic Regression (penalty='{penalty}', C={C})")
    return avaliar_modelo(model, X_test, y_test, f"Logistic Regression ({penalty}-{solver}-{C})")

In [11]:
for C in [0.01, 0.1, 0.5, 1.0, 2.0]:
    try_logistic_regression('l1', C, class_weight='balanced', solver='liblinear')

Logistic Regression (penalty='l1', C=0.01)
Logistic Regression (l1-liblinear-0.01) -> AUC-ROC: 0.7539 | Brier Score: 0.2036
Logistic Regression (penalty='l1', C=0.1)
Logistic Regression (l1-liblinear-0.1) -> AUC-ROC: 0.7628 | Brier Score: 0.1979
Logistic Regression (penalty='l1', C=0.5)
Logistic Regression (l1-liblinear-0.5) -> AUC-ROC: 0.7630 | Brier Score: 0.1974
Logistic Regression (penalty='l1', C=1.0)
Logistic Regression (l1-liblinear-1.0) -> AUC-ROC: 0.7631 | Brier Score: 0.1974
Logistic Regression (penalty='l1', C=2.0)
Logistic Regression (l1-liblinear-2.0) -> AUC-ROC: 0.7631 | Brier Score: 0.1974


### Ridge

In [12]:
for C in [0.01, 0.1, 0.5, 1.0, 2.0]:
    try_logistic_regression('l2', C, solver='lbfgs', class_weight='balanced')

Logistic Regression (penalty='l2', C=0.01)
Logistic Regression (l2-lbfgs-0.01) -> AUC-ROC: 0.7601 | Brier Score: 0.1980
Logistic Regression (penalty='l2', C=0.1)
Logistic Regression (l2-lbfgs-0.1) -> AUC-ROC: 0.7621 | Brier Score: 0.1975
Logistic Regression (penalty='l2', C=0.5)
Logistic Regression (l2-lbfgs-0.5) -> AUC-ROC: 0.7629 | Brier Score: 0.1974
Logistic Regression (penalty='l2', C=1.0)
Logistic Regression (l2-lbfgs-1.0) -> AUC-ROC: 0.7628 | Brier Score: 0.1974
Logistic Regression (penalty='l2', C=2.0)
Logistic Regression (l2-lbfgs-2.0) -> AUC-ROC: 0.7623 | Brier Score: 0.1974


In [13]:
for C in [0.01, 0.1, 0.5, 1.0, 2.0]:
    try_logistic_regression('l2', C, solver='liblinear', class_weight='balanced')

Logistic Regression (penalty='l2', C=0.01)
Logistic Regression (l2-liblinear-0.01) -> AUC-ROC: 0.7600 | Brier Score: 0.1987
Logistic Regression (penalty='l2', C=0.1)
Logistic Regression (l2-liblinear-0.1) -> AUC-ROC: 0.7622 | Brier Score: 0.1976
Logistic Regression (penalty='l2', C=0.5)
Logistic Regression (l2-liblinear-0.5) -> AUC-ROC: 0.7630 | Brier Score: 0.1974
Logistic Regression (penalty='l2', C=1.0)
Logistic Regression (l2-liblinear-1.0) -> AUC-ROC: 0.7632 | Brier Score: 0.1974
Logistic Regression (penalty='l2', C=2.0)
Logistic Regression (l2-liblinear-2.0) -> AUC-ROC: 0.7630 | Brier Score: 0.1974


### Elastic Net 

In [ ]:
def try_elastic_net(C=1.0, l1_ratio=0.5, class_weight='balanced', solver='saga', max_iter=1000):
    model = LogisticRegression(
        penalty='elasticnet', 
        C=C, 
        l1_ratio=l1_ratio, 
        solver=solver, 
        class_weight=class_weight, 
        max_iter=max_iter, 
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    return avaliar_modelo(model, X_test, y_test, f"Logistic Regression (C={C}, l1_ratio={l1_ratio})")

In [18]:
for C in [0.01, 0.1, 0.5, 1.0, 2.0]:
    try_elastic_net(C=C, l1_ratio=0.5, class_weight='balanced', solver='saga', max_iter=5000)

c:\Repos\canine-mammary-tumors\.venv\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Logistic Regression (C=0.01, l1_ratio=0.5)
Logistic Regression (C=0.01, l1_ratio=0.5) -> AUC-ROC: 0.7199 | Brier Score: 0.2200


c:\Repos\canine-mammary-tumors\.venv\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Logistic Regression (C=0.1, l1_ratio=0.5)
Logistic Regression (C=0.1, l1_ratio=0.5) -> AUC-ROC: 0.7497 | Brier Score: 0.3591


c:\Repos\canine-mammary-tumors\.venv\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Logistic Regression (C=0.5, l1_ratio=0.5)
Logistic Regression (C=0.5, l1_ratio=0.5) -> AUC-ROC: 0.7323 | Brier Score: 0.1990


c:\Repos\canine-mammary-tumors\.venv\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Logistic Regression (C=1.0, l1_ratio=0.5)
Logistic Regression (C=1.0, l1_ratio=0.5) -> AUC-ROC: 0.7016 | Brier Score: 0.2934
Logistic Regression (C=2.0, l1_ratio=0.5)
Logistic Regression (C=2.0, l1_ratio=0.5) -> AUC-ROC: 0.7291 | Brier Score: 0.2244


c:\Repos\canine-mammary-tumors\.venv\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [19]:
for L1_RATIO in [0.25, 0.50, 0.75]:
    try_elastic_net(C=0.1, l1_ratio=L1_RATIO, class_weight='balanced', solver='saga', max_iter=10000)

c:\Repos\canine-mammary-tumors\.venv\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Logistic Regression (C=0.1, l1_ratio=0.25)
Logistic Regression (C=0.1, l1_ratio=0.25) -> AUC-ROC: 0.7193 | Brier Score: 0.2636


c:\Repos\canine-mammary-tumors\.venv\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Logistic Regression (C=0.1, l1_ratio=0.5)
Logistic Regression (C=0.1, l1_ratio=0.5) -> AUC-ROC: 0.6330 | Brier Score: 0.1413
Logistic Regression (C=0.1, l1_ratio=0.75)
Logistic Regression (C=0.1, l1_ratio=0.75) -> AUC-ROC: 0.7513 | Brier Score: 0.3680


c:\Repos\canine-mammary-tumors\.venv\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


## Modelos de Árvores

### Random Forest

#### Parâmetros

- n_estimators -> número de árvores (quantidade de decisores)
    - Geralmente, quanto mais, melhor a performance e mais estável é o modelo. No entanto, o custo computacional aumenta (demora mais para treinar). Chega um ponto em que adicionar mais árvores não melhora o modelo (retorno decrescente).

- max_depth -> A profundidade máxima da árvore (tomar cuidado com overfiting)
    - Controla o número máximo de "níveis" ou perguntas que a árvore pode fazer antes de dar uma resposta. Se for None, a árvore cresce até que todas as folhas sejam "puras" (contenham apenas uma classe).

- min_samples_split -> O número mínimo de amostras necessárias para dividir um nó interno.
    - Se um nó tem 5 amostras e você definiu esse valor como 10, a árvore para de dividir ali, mesmo que ainda não seja uma folha pura. Aumentar esse valor (ex: 5, 10, 20) força a árvore a ser mais generalista e menos sensível a pequenos ruídos nos dados.

- min_samples_leaf -> O número mínimo de amostras que uma folha (o nó final que dá a decisão) deve ter.
    - Semelhante ao min_samples_split, mas foca no final da árvore. Se uma divisão fosse criar uma folha com apenas 1 amostra, a divisão não acontece. Aumentar esse valor (ex: 2, 5, 10) também previne overfitting, suavizando o modelo.

- max_leaf_nodes -> O número máximo de folhas que a árvore pode ter.
    - É uma forma alternativa de limitar o crescimento, focando no número total de resultados possíveis por árvore.

- max_features -> O número máximo de características (features) que o modelo pode considerar ao procurar a melhor divisão em cada nó.
    - 'sqrt': Usa a raiz quadrada do número total de features (o padrão e geralmente o melhor para classificação).
    - 'log2': Usa o logaritmo na base 2.
    - None: Usa todas as features (nesse caso, vira uma técnica chamada Bagging e perde um pouco da vantagem do Random Forest).

- bootstrap -> Se as amostras são retiradas com reposição (Bootstrap) ao construir as árvores.
    - Em vez de treinar cada árvore com os mesmos 100% dos dados, cada árvore recebe um subconjunto aleatório (com o mesmo tamanho do original, mas algumas linhas são repetidas e outras ignoradas). Isso é fundamental para a diversidade das árvores.

- max_samples (Padrão = None) -> (Usado apenas se bootstrap=True). A porcentagem máxima de amostras a ser usada para treinar cada árvore. Se você tiver 1 milhão de linhas, pode querer que cada árvore use apenas 50% delas para treinar mais rápido, sem perder a diversidade.

- class_weigt -> peso das classes


#### Referências

https://www.mdpi.com/1424-8220/22/17/6709 -> base 1
- RF	max depth: 190, n estimators: 709, random state: 50
- XGB	learning rate: 0.068, max depth: 11, n estimators: 100, random state: 50

https://www.mdpi.com/2076-3417/12/14/7253 -> base 2
- XG boost
    - parameter   Search Type     Search Space Range
    - gamma	continuous	[5.0, 11.0]
    - learning rate	continuous	[0.07, 0.6]
    - n estimators	discrete	50, 100, 150
    - regulation alpha	discrete	1 × 10^−5, 1 × 10^−2 ,0.75
    - regulation lambda	discrete	1 × 10^−5, 1 × 10^−2 ,0.45
    - min child weight	discrete	1.5, 6, 10
    - subsample	discrete	0.6, 0.95
    - max depth	discrete	3, 6, 9

- RF
    - parameter   Search Type     Search Space Range
    - n estimators	discrete	100, 200, 500
    - max depth	discrete	4, 5, 6, ..., 11
    - min samples	discrete	2, 3, 4, 5
    - min samples leaf	discrete	1, 3, 5, 7

In [22]:
# Base 1
RANDOM_STATE_1 = 50
CLASS_WEIGHT_1 = 'balanced'
MAX_DEPTH_1 = 190
N_ESTIMATORS_1 = 709

# Base 2
N_ESTIMATORS_2 = [100, 200, 500]
MAX_DEPTH_2 = [4,5,6,7,8,9,10,11]
MIN_SAMPLES_2 = [2, 3, 4, 5]
MIN_SAMPLES_LEAF_2 = [1, 3, 5, 7]
CLASS_WEIGHT_2 = 'balanced'

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_base1 = RandomForestClassifier(
    n_estimators=N_ESTIMATORS_1,
    random_state=RANDOM_STATE_1,
    max_depth=MAX_DEPTH_1
)
rf_base1.fit(X_train, y_train)

avaliar_modelo(rf_base1, X_test, y_test, "Random Forest - Base")

Random Forest - Base -> AUC-ROC: 0.7631 | Brier Score: 0.0250


In [19]:
from sklearn.ensemble import RandomForestClassifier

rf_base1_balanced = RandomForestClassifier(
    n_estimators=N_ESTIMATORS_1,
    class_weight=CLASS_WEIGHT_1,
    random_state=RANDOM_STATE_1,
    max_depth=MAX_DEPTH_1
)
rf_base1_balanced.fit(X_train, y_train)

avaliar_modelo(rf_base1_balanced, X_test, y_test, "Random Forest - Base (class balanced)")

Random Forest - Base (class balanced) -> AUC-ROC: 0.7606 | Brier Score: 0.1891


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': N_ESTIMATORS_2,     
    'max_depth': MAX_DEPTH_2,     
    'min_samples_split': MIN_SAMPLES_2,     
    'min_samples_leaf': MIN_SAMPLES_LEAF_2        
}


rf_base2 = RandomForestClassifier(random_state=42, class_weight=CLASS_WEIGHT_2)

grid_search = GridSearchCV(
    estimator=rf_base2, 
    param_grid=param_grid, 
    scoring='roc_auc', 
    n_jobs=-1,
    verbose=2 
)
grid_search.fit(X_train, y_train)


print("\n--- RESULTADOS DO GRID SEARCH ---")
print(f"Melhor nota AUC-ROC durante o treino: {grid_search.best_score_:.3f}")
print("Melhores parâmetros encontrados:")
print(grid_search.best_params_)

melhor_rf = grid_search.best_estimator_